In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import LinearRegression

In [2]:
df_X4 = pd.read_csv("homework_4.1.csv")

In [3]:
# ── Method 1: Simple ratio of averages ──────────────────────────────────────
mean_Y_Z1 = df_X4.loc[df_X4['Z']==1, 'Y'].mean()
mean_Y_Z0 = df_X4.loc[df_X4['Z']==0, 'Y'].mean()
mean_X_Z1 = df_X4.loc[df_X4['Z']==1, 'X'].mean()
mean_X_Z0 = df_X4.loc[df_X4['Z']==0, 'X'].mean()

diff_Y = mean_Y_Z1 - mean_Y_Z0
diff_X = mean_X_Z1 - mean_X_Z0
iv_method1 = diff_Y / diff_X
# → ~1.562

# ── Method 2: Local ratios averaged over narrow bins of W ───────────────────
n_bins = 50
df_X4['W_bin'] = pd.cut(df_X4['W'], bins=n_bins)

local_effects = []
for bin_label, group in df_X4.groupby('W_bin', observed=True):
    g1 = group[group['Z']==1]
    g0 = group[group['Z']==0]
    if len(g1) > 1 and len(g0) > 1:
        dy = g1['Y'].mean() - g0['Y'].mean()
        dx = g1['X'].mean() - g0['X'].mean()
        if abs(dx) > 1e-8:
            local_effects.append(dy / dx)

iv_method2 = np.mean(local_effects)
# → ~1.485
#The two methods agree reasonably well (~1.56 vs ~1.48), with the small difference due to Method 2 controlling for W within each bin before averaging — which is the more robust estimate when W is a true confounder.Want to be notified when C

In [4]:
print(iv_method1)

1.5618587073765746


In [5]:
print(iv_method2)

1.4848406457567933


In [6]:
df_X4a = pd.read_csv("homework_4.2.a.csv")
df_X4b = pd.read_csv("homework_4.2.b.csv")

In [8]:
df_X4a.columns = ['X', 'Y']
df_X4b.columns = ['X', 'Y']

cutoff = 80

for df, name in [(df_X4a, 'A'), (df_X4b, 'B')]:
    below = df[df.X < cutoff]
    above = df[df.X >= cutoff]

    # Trend below cutoff
    slope, intercept, r, p, se = stats.linregress(below.X, below.Y)
    print(f'Dataset {name} slope below cutoff: {slope:.6f}, p={p:.4f}')
    print(f'Direction: {"INCREASING" if slope > 0 else "DECREASING"}')

    # Jump at cutoff
    near_below = df[(df.X >= 75) & (df.X < 80)].Y.mean()
    near_above = df[(df.X >= 80) & (df.X < 85)].Y.mean()
    jump = near_above - near_below
    print(f'Jump at cutoff: {jump:.4f} => course {"HELPS" if jump > 0 else "HURTS"}')

Dataset A slope below cutoff: 0.000224, p=0.5117
Direction: INCREASING
Jump at cutoff: 0.2966 => course HELPS
Dataset B slope below cutoff: 0.010217, p=0.0000
Direction: INCREASING
Jump at cutoff: 0.2386 => course HELPS


In [11]:
CUTOFF = 80

def rdd_analysis(df, x_col, y_col, cutoff, name):
    below = df[df[x_col] < cutoff].copy()
    above = df[df[x_col] >= cutoff].copy()

    def slope(data, x, y):
        X = data[[x]].values
        Y = data[y].values
        model = LinearRegression().fit(X, Y)
        return model.coef_[0], model.intercept_

    slope_below, int_below = slope(below, x_col, y_col)
    slope_above, int_above = slope(above, x_col, y_col)

    pred_below = int_below + slope_below * cutoff
    pred_above = int_above + slope_above * cutoff
    jump = pred_above - pred_below

    print(f"Dataset {name}")
    print(f"  Slope below cutoff: {slope_below:.6f}")
    print(f"  Slope above cutoff: {slope_above:.6f}")
    print(f"  Jump at cutoff:     {jump:.4f}")
    print(f"  Course effect:      {'HELPS' if jump > 0 else 'NO EFFECT'}")
    print(f"  Slope after is:     {'HIGHER' if slope_above > slope_below else 'LOWER'}")

rdd_analysis(df_X4a, 'X',  'Y',  CUTOFF, 'A')
rdd_analysis(df_X4b, 'X', 'Y', CUTOFF, 'B')

#Bonus: Does the math course help? (for both datasets)
#Both datasets show a positive jump at the cutoff (X = 80), meaning students just above the threshold — who were assigned to the math course — had significantly higher college admission rates than those just below:

#Dataset A: Jump ≈ +0.296 (≈ 30 percentage point increase) ✅ Course helps
#Dataset B: Jump ≈ +0.197 (≈ 20 percentage point increase) ✅ Course helps
# Want to be notified when Claude responds?

Dataset A
  Slope below cutoff: 0.000224
  Slope above cutoff: 0.000161
  Jump at cutoff:     0.2958
  Course effect:      HELPS
  Slope after is:     LOWER
Dataset B
  Slope below cutoff: 0.010217
  Slope above cutoff: 0.005009
  Jump at cutoff:     0.1971
  Course effect:      HELPS
  Slope after is:     LOWER
